In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — go to Runtime > Change runtime type > T4 GPU")

!pip install transformers datasets scikit-learn pandas numpy matplotlib seaborn -q

import transformers
print("Transformers version:", transformers.__version__)

In [ ]:
!pip install kaggle -q
from google.colab import files
files.upload()  # upload your kaggle.json API key here

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d jp797498e/twitter-entity-sentiment-analysis --unzip


In [ ]:
import pandas as pd
from transformers import DistilBertTokenizer
from torch.utils.data import Dataset, DataLoader
import torch

# Load dataset
# Upload the Kaggle CSV first (twitter_training.csv / twitter_validation.csv)

train_df = pd.read_csv("twitter_training.csv", header=None,
                        names=["id", "entity", "sentiment", "text"])
val_df   = pd.read_csv("twitter_validation.csv", header=None,
                        names=["id", "entity", "sentiment", "text"])

label_map = {"Positive": 2, "Neutral": 1, "Negative": 0, "Irrelevant": 3}
for df in [train_df, val_df]:
    df.dropna(subset=["text", "sentiment"], inplace=True)
    df["label"] = df["sentiment"].map(label_map)

print(train_df["sentiment"].value_counts())

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(texts, max_len=64):
    return tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

class TweetDataset(Dataset):
    def __init__(self, df):
        encoded = tokenize(df["text"])
        self.input_ids      = encoded["input_ids"]
        self.attention_mask = encoded["attention_mask"]
        self.labels         = torch.tensor(df["label"].values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }

train_dataset = TweetDataset(train_df)
val_dataset   = TweetDataset(val_df)

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

sample = train_dataset[0]
print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["labels"].item())

In [ ]:
# Task 3.3d — FULL BERT TRAINING RUN (GPU)
# Identical setup to DistilBERT for fair comparison

import torch, gc, time
from transformers import BertTokenizer, BertForSequenceClassification, get_scheduler
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# ── Free memory ──────────────────────────────────────────────
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── Load & prep data (in case this is a fresh runtime) ───────
label_map = {"Positive": 2, "Neutral": 1, "Negative": 0, "Irrelevant": 3}

train_df = pd.read_csv("twitter_training.csv", header=None,
                       names=["id", "entity", "sentiment", "text"])
val_df = pd.read_csv("twitter_validation.csv", header=None,
                     names=["id", "entity", "sentiment", "text"])

for df in [train_df, val_df]:
    df.dropna(subset=["text", "sentiment"], inplace=True)
    df["label"] = df["sentiment"].map(label_map)

# ── BERT tokenizer (different vocabulary than DistilBERT) ────
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(texts, max_len=64):
    return tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

class TweetDataset(Dataset):
    def __init__(self, df):
        encoded = tokenize(df["text"])
        self.input_ids      = encoded["input_ids"]
        self.attention_mask = encoded["attention_mask"]
        self.labels         = torch.tensor(df["label"].values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }

train_dataset = TweetDataset(train_df)
val_dataset   = TweetDataset(val_df)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# ── Settings (IDENTICAL to DistilBERT for fairness) ──────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_LABELS  = 4
EPOCHS      = 3
BATCH_SIZE  = 32              # smaller than DistilBERT — BERT is 2× larger
LR          = 2e-5

print(f"Device: {DEVICE}")
assert DEVICE.type == "cuda", "STOP — GPU not active. Fix runtime before continuing."

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

print(f"Batches per epoch: {len(train_loader)}")

# ── Fresh BERT model ─────────────────────────────────────────
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=NUM_LABELS
).to(DEVICE)

print(f"Model on: {next(model.parameters()).device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
num_steps = len(train_loader) * EPOCHS
scheduler = get_scheduler("linear", optimizer=optimizer,
                           num_warmup_steps=num_steps // 10,
                           num_training_steps=num_steps)

# ── Training loop ────────────────────────────────────────────
history = {"train_loss": [], "val_acc": [], "val_f1_w": [], "val_f1_m": []}
class_names = ["Negative", "Neutral", "Positive", "Irrelevant"]

total_start = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        outputs = model(**batch)
        outputs.loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += outputs.loss.item()

        if (step + 1) % 100 == 0:
            print(f"  Epoch {epoch+1} | batch {step+1}/{len(train_loader)} | running loss: {total_loss/(step+1):.4f}")

    avg_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            logits = model(**batch).logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1_w = f1_score(all_labels, all_preds, average="weighted")
    f1_m = f1_score(all_labels, all_preds, average="macro")

    history["train_loss"].append(avg_loss)
    history["val_acc"].append(acc)
    history["val_f1_w"].append(f1_w)
    history["val_f1_m"].append(f1_m)

    epoch_time = time.time() - epoch_start
    print(f"\nEpoch {epoch+1}/{EPOCHS} ({epoch_time:.1f}s) | "
          f"Loss: {avg_loss:.4f} | Acc: {acc:.4f} | Weighted F1: {f1_w:.4f} | Macro F1: {f1_m:.4f}\n")

total_time = time.time() - total_start
print(f"=== Total BERT training time: {total_time/60:.1f} min ===")

# ── Prediction counts ────────────────────────────────────────
pred_counts = np.bincount(all_preds, minlength=4)
print("\nPrediction counts per class:")
for name, count in zip(class_names, pred_counts):
    print(f"  {name}: {count}")

# ── Confusion matrix ─────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("BERT (Full Training) — Confusion Matrix")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))